# Step 5b: RAG Framework — Retrieval, LLM & Agent
## YouTube Fast Fashion Intelligence Engine — CSCI370

### What this notebook covers

| Component | What it does |
|---|---|
| Metadata Retrieval | Filter comments by sentiment, video, topic |
| Semantic Retrieval | Find comments by meaning similarity |
| Lexical Retrieval (BM25) | Find comments by exact keyword match |
| Hybrid Retrieval | Combine semantic + lexical via RRF |
| Query Analysis | Understand what the user is asking |
| LLM Q&A | Answer questions grounded in retrieved comments |
| LLM Summarization | Summarize a set of retrieved comments |
| Agent Orchestration | Route each query to the right tool |

### End to end flow
```
User question -> Query Analysis -> Route to retrieval method
-> Retrieve top N comments -> Pass to Groq LLaMA 3 -> Grounded answer
```


## 0. Install Libraries

In [ ]:
!pip install chromadb rank_bm25 groq scikit-learn -q

## 1. Load Everything from Step 5a

In [ ]:
import pandas as pd
import numpy as np
import chromadb
import pickle
from groq import Groq
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('youtube_comments_topics.csv')
df = df.dropna(subset=['text_clean']).reset_index(drop=True)

with open('embedding_function.pkl', 'rb') as f:
    embedding_fn = pickle.load(f)

client = chromadb.PersistentClient(path='./chroma_db')
collection = client.get_collection('youtube_comments', embedding_function=embedding_fn)

with open('bm25_index.pkl', 'rb') as f:
    bm25_data = pickle.load(f)

bm25_index   = bm25_data['index']
bm25_texts   = bm25_data['texts']
bm25_indices = bm25_data['indices']

print(f"Dataset     : {len(df):,} comments")
print(f"ChromaDB    : {collection.count():,} documents")
print(f"BM25 corpus : {len(bm25_texts):,} documents")
print("All loaded!")

In [ ]:
# Colab Secrets stores the key — click the key icon in the left sidebar,
# add a secret named GROQ_API_KEY, and turn on notebook access.
from google.colab import userdata

GROQ_API_KEY = userdata.get('GROQ_API_KEY')
groq_client = Groq(api_key=GROQ_API_KEY)

test = groq_client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[{"role": "user", "content": "Say Groq connected and nothing else."}],
    max_tokens=10
)
print(test.choices[0].message.content)

## 3. The Four Retrieval Methods

Each method has different strengths. The agent decides which to use per query.


### Method 1: Metadata Filtering
Filter by stored properties. No query needed.

In [ ]:
def metadata_retrieval(sentiment=None, video_id=None, topic=None, n=10):
    # Filter comments by their metadata fields
    # Use when: 'show me all negative comments', 'comments from video X'
    where_clause = {}
    if sentiment:
        where_clause['sentiment_label'] = sentiment
    if video_id:
        where_clause['video_id'] = video_id
    if topic is not None:
        where_clause['topic_general'] = topic

    if not where_clause:
        return df.nlargest(n, 'like_count')[['text_clean','sentiment_label','like_count']].to_dict('records')

    results = collection.get(where=where_clause, limit=n)
    return [{'text': doc, **meta} for doc, meta in zip(results['documents'], results['metadatas'])]

# Test
print("METADATA RETRIEVAL - Negative comments:")
for r in metadata_retrieval(sentiment='Negative', n=3):
    print(f"  [{r.get('sentiment_label')}] {str(r.get('text', r.get('text_clean','')))[:120]}")

### Method 2: Semantic Search
Find comments by MEANING similarity. Works even with different words.

In [ ]:
def semantic_retrieval(query, n=10, sentiment=None):
    # Find comments similar in meaning to the query
    # Use when: 'what do people think about environmental damage?'
    kwargs = {'query_texts': [query], 'n_results': n}
    if sentiment:
        kwargs['where'] = {'sentiment_label': sentiment}

    results = collection.query(**kwargs)
    return [{'text': doc, **meta} for doc, meta in zip(results['documents'][0], results['metadatas'][0])]

# Test
print("SEMANTIC RETRIEVAL - 'worker exploitation and low wages':")
for r in semantic_retrieval("worker exploitation and low wages", n=3):
    print(f"  [{r['sentiment_label']}] {r['text'][:150]}")

### Method 3: Lexical Search (BM25)
Find comments by EXACT keywords. Best for brand names and specific terms.

In [ ]:
def lexical_retrieval(query, n=10):
    # Find comments by exact keyword matching using BM25
    # Use when: 'find comments about Temu', 'H&M specifically'
    # BM25 ranks by how often and how uniquely the query words appear
    tokens = query.lower().split()
    scores = bm25_index.get_scores(tokens)
    top_indices = np.argsort(scores)[::-1][:n]

    results = []
    for idx in top_indices:
        if scores[idx] > 0:
            orig_idx = bm25_indices[idx]
            row = df.iloc[orig_idx]
            results.append({
                'text'           : bm25_texts[idx],
                'bm25_score'     : float(scores[idx]),
                'sentiment_label': str(row.get('sentiment_label', '')),
                'like_count'     : int(row.get('like_count', 0)),
            })
    return results

# Test
print("LEXICAL RETRIEVAL - 'Temu cheap prices app':")
for r in lexical_retrieval("Temu cheap prices app", n=3):
    print(f"  [BM25: {r['bm25_score']:.2f}] [{r['sentiment_label']}] {r['text'][:150]}")

### Method 4: Hybrid Search
Combine semantic + lexical using **Reciprocal Rank Fusion (RRF)**.

RRF formula: score = sum(1 / (rank + 60)) for each list the document appears in.
Documents ranking well in BOTH lists get the highest scores.
This is the best general-purpose method.


In [ ]:
def hybrid_retrieval(query, n=10, sentiment=None, semantic_weight=0.6, lexical_weight=0.4):
    # Best general method: combines semantic + lexical via RRF
    semantic_results = semantic_retrieval(query, n=n*2, sentiment=sentiment)
    lexical_results  = lexical_retrieval(query, n=n*2)

    rrf_scores = {}
    doc_data   = {}

    for rank, result in enumerate(semantic_results):
        key = result['text'][:100]
        rrf_scores[key] = rrf_scores.get(key, 0) + semantic_weight * (1 / (rank + 60))
        doc_data[key]   = result

    for rank, result in enumerate(lexical_results):
        key = result['text'][:100]
        rrf_scores[key] = rrf_scores.get(key, 0) + lexical_weight * (1 / (rank + 60))
        if key not in doc_data:
            doc_data[key] = result

    sorted_keys = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:n]
    return [doc_data[k] for k in sorted_keys]

# Test
print("HYBRID RETRIEVAL - 'Shein environmental damage pollution':")
for r in hybrid_retrieval("Shein environmental damage pollution", n=3):
    print(f"  [{r.get('sentiment_label','?')}] {r.get('text','')[:150]}")

## 4. Query Analysis

Before retrieving, we analyze the query to decide:
- Which retrieval strategy to use
- Whether to apply a sentiment filter
- Whether to Q&A or summarize

| Question | Strategy |
|---|---|
| "What do negative comments say?" | Metadata |
| "Find comments about Temu" | Lexical |
| "How do people feel about the environment?" | Semantic |
| "What are the main concerns?" | Hybrid |
| "Summarize the discussion" | Hybrid + summarize |


In [ ]:
def analyze_query(query):
    q = query.lower()

    # Detect sentiment filter
    sentiment_filter = None
    if any(w in q for w in ['negative','criticism','complain','hate','angry','upset','bad']):
        sentiment_filter = 'Negative'
    elif any(w in q for w in ['positive','praise','love','good','great','appreciate']):
        sentiment_filter = 'Positive'
    elif 'neutral' in q:
        sentiment_filter = 'Neutral'

    # Detect task
    task = 'summarize' if any(w in q for w in ['summarize','summary','overview','recap']) else 'qa'

    # Detect retrieval strategy
    brands = ['shein','temu','zara','h&m','primark','asos','boohoo','uniqlo','romwe']
    if any(b in q for b in brands) and len(q.split()) <= 5:
        strategy = 'lexical'
    elif any(w in q for w in ['feel','think','opinion','concern','worry','impact','environment']):
        strategy = 'semantic'
    else:
        strategy = 'hybrid'

    return {'strategy': strategy, 'sentiment_filter': sentiment_filter,
            'task': task, 'original_query': query}

# Test
test_queries = [
    "What do negative comments say about Shein?",
    "Summarize concerns about environmental damage",
    "Find comments about Temu",
    "How do people feel about fast fashion workers?",
]
print("QUERY ANALYSIS:")
print("-" * 60)
for q in test_queries:
    a = analyze_query(q)
    print(f"Query    : {q}")
    print(f"Strategy : {a['strategy']} | Sentiment: {a['sentiment_filter']} | Task: {a['task']}")
    print()

## 5. LLM Q&A and Summarization

We pass retrieved comments to Groq LLaMA 3 as context.
The LLM is instructed to answer ONLY from those comments.
This prevents hallucination.


In [ ]:
def llm_qa(query, retrieved_comments, max_comments=8):
    context = ""
    for i, c in enumerate(retrieved_comments[:max_comments], 1):
        text      = c.get('text', c.get('text_clean', ''))
        sentiment = c.get('sentiment_label', 'Unknown')
        likes     = c.get('like_count', 0)
        context  += f"[Comment {i} | Sentiment: {sentiment} | Likes: {likes}]\n{text}\n\n"

    prompt = (
        "You are an analyst studying YouTube comments about fast fashion brands.\n\n"
        "Answer the question below based ONLY on the YouTube comments provided.\n"
        "Do not use outside knowledge. Reference what commenters actually said.\n"
        "Keep your answer to 3-5 sentences.\n\n"
        f"QUESTION: {query}\n\n"
        f"YOUTUBE COMMENTS:\n{context}\nANSWER:"
    )

    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=400,
        temperature=0.3
    )
    return response.choices[0].message.content.strip()


def llm_summarize(retrieved_comments, topic_description="fast fashion", max_comments=10):
    context = "".join([
        f"[Comment {i} | {c.get('sentiment_label','?')}]: {c.get('text', c.get('text_clean',''))}\n\n"
        for i, c in enumerate(retrieved_comments[:max_comments], 1)
    ])

    prompt = (
        f"Summarize these YouTube comments about {topic_description} in 5 sentences.\n"
        "Cover: main themes, overall sentiment, strongest concerns or praises, recurring patterns.\n\n"
        f"COMMENTS:\n{context}\nSUMMARY:"
    )

    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=500,
        temperature=0.3
    )
    return response.choices[0].message.content.strip()

print("LLM functions ready!")

## 6. Agent Orchestration

The agent ties all components together:
1. Query analysis -> decide strategy and filters
2. Route to the right retrieval method
3. Call the LLM with retrieved context
4. Return grounded answer + source comments


In [ ]:
def rag_agent(query, n_results=8):
    print(f"\nProcessing: '{query}'")
    print("-" * 60)

    analysis  = analyze_query(query)
    strategy  = analysis['strategy']
    sentiment = analysis['sentiment_filter']
    task      = analysis['task']

    print(f"Strategy: {strategy} | Sentiment: {sentiment} | Task: {task}")

    if strategy == 'lexical':
        retrieved = lexical_retrieval(query, n=n_results)
    elif strategy == 'semantic':
        retrieved = semantic_retrieval(query, n=n_results, sentiment=sentiment)
    elif strategy == 'metadata':
        retrieved = metadata_retrieval(sentiment=sentiment, n=n_results)
    else:
        retrieved = hybrid_retrieval(query, n=n_results, sentiment=sentiment)

    print(f"Retrieved: {len(retrieved)} comments")

    answer = llm_summarize(retrieved, topic_description=query) if task == 'summarize' else llm_qa(query, retrieved)

    return {
        'query'          : query,
        'analysis'       : analysis,
        'retrieved_count': len(retrieved),
        'answer'         : answer,
        'source_comments': retrieved[:3]
    }

print("RAG agent ready!")

In [ ]:
# Test the full agent
test_questions = [
    "What do people say about Shein's labor practices?",
    "Summarize the negative comments about fast fashion environmental impact",
    "How do positive commenters justify buying from Temu?",
]

for question in test_questions:
    result = rag_agent(question)
    print(f"\nQUESTION : {result['query']}")
    print(f"\nANSWER:")
    print(result['answer'])
    if result['source_comments']:
        src = result['source_comments'][0]
        print(f"\nTop source: [{src.get('sentiment_label','?')}] {src.get('text','')[:180]}")
    print("\n" + "="*65)

## 7. Save RAG Pipeline for the Dashboard

In [ ]:
rag_code = '''
import numpy as np
import chromadb
import pickle
from groq import Groq

def load_rag_pipeline(groq_api_key, db_path="./chroma_db",
                      embedding_pkl="embedding_function.pkl",
                      bm25_pkl="bm25_index.pkl"):
    with open(embedding_pkl, "rb") as f:
        embedding_fn = pickle.load(f)
    client = chromadb.PersistentClient(path=db_path)
    collection = client.get_collection("youtube_comments", embedding_function=embedding_fn)
    with open(bm25_pkl, "rb") as f:
        bm25_data = pickle.load(f)
    groq_client = Groq(api_key=groq_api_key)
    return collection, bm25_data, groq_client

def rag_query(query, collection, bm25_data, groq_client, n_results=8):
    bm25_index   = bm25_data["index"]
    bm25_texts   = bm25_data["texts"]
    bm25_indices = bm25_data["indices"]
    q = query.lower()

    sentiment_filter = None
    if any(w in q for w in ["negative","complain","hate","bad","angry"]):
        sentiment_filter = "Negative"
    elif any(w in q for w in ["positive","love","great","good","praise"]):
        sentiment_filter = "Positive"

    task = "summarize" if any(w in q for w in ["summarize","summary","overview"]) else "qa"

    brands = ["shein","temu","zara","h&m","primark","asos","boohoo"]
    if any(b in q for b in brands) and len(q.split()) <= 5:
        strategy = "lexical"
    elif any(w in q for w in ["feel","think","concern","impact","environment"]):
        strategy = "semantic"
    else:
        strategy = "hybrid"

    if strategy == "lexical":
        scores   = bm25_index.get_scores(q.split())
        top_idxs = np.argsort(scores)[::-1][:n_results]
        docs     = [{"text": bm25_texts[i], "sentiment_label": "", "like_count": 0} for i in top_idxs if scores[i] > 0]
    elif strategy == "semantic":
        kwargs = {"query_texts": [query], "n_results": n_results}
        if sentiment_filter:
            kwargs["where"] = {"sentiment_label": sentiment_filter}
        res  = collection.query(**kwargs)
        docs = [{"text": d, **m} for d, m in zip(res["documents"][0], res["metadatas"][0])]
    else:
        kwargs = {"query_texts": [query], "n_results": n_results * 2}
        if sentiment_filter:
            kwargs["where"] = {"sentiment_label": sentiment_filter}
        sem = collection.query(**kwargs)
        sem_docs = [{"text": d, **m} for d, m in zip(sem["documents"][0], sem["metadatas"][0])]
        lex_scrs = bm25_index.get_scores(q.split())
        lex_top  = np.argsort(lex_scrs)[::-1][:n_results * 2]
        lex_docs = [{"text": bm25_texts[i], "sentiment_label": "", "like_count": 0} for i in lex_top]
        seen, docs = set(), []
        for doc in sem_docs + lex_docs:
            key = doc["text"][:80]
            if key not in seen:
                seen.add(key)
                docs.append(doc)
            if len(docs) >= n_results:
                break

    context = "".join([
        f"[Comment {i} | {d.get(\'sentiment_label\',\'?\')}]: {d[\'text\']}\\n\\n"
        for i, d in enumerate(docs[:8], 1)
    ])

    if task == "summarize":
        prompt = f"Summarize these YouTube comments about fast fashion in 5 sentences:\\n\\n{context}\\nSUMMARY:"
    else:
        prompt = (
            f"Answer based ONLY on these comments. Be concise (3-5 sentences).\\n\\n"
            f"Question: {query}\\n\\nComments:\\n{context}\\nAnswer:"
        )

    resp = groq_client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=400,
        temperature=0.3
    )
    return {
        "answer"  : resp.choices[0].message.content.strip(),
        "sources" : docs[:3],
        "strategy": strategy,
        "task"    : task
    }
'''

with open('rag_pipeline.py', 'w') as f:
    f.write(rag_code)
print("rag_pipeline.py saved - dashboard will import this.")

## Summary

**What we built:**

| Component | Function | Purpose |
|---|---|---|
| Metadata Retrieval | `metadata_retrieval()` | Filter by fields |
| Semantic Retrieval | `semantic_retrieval()` | Meaning similarity |
| Lexical Retrieval | `lexical_retrieval()` | Exact keywords (BM25) |
| Hybrid Retrieval | `hybrid_retrieval()` | Semantic + Lexical via RRF |
| Query Analysis | `analyze_query()` | Route to right tool |
| LLM Q&A | `llm_qa()` | Grounded answers |
| LLM Summarization | `llm_summarize()` | Comment summaries |
| Agent | `rag_agent()` | Orchestrates everything |
